In [3]:
from langchain_classic.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableMap

In [8]:
docs = TextLoader("/Users/pawanpahune/RAG_From_Scratch/Query_Enhancement_Techniques/Langchain_crewai.txt").load()
splitter = RecursiveCharacterTextSplitter(chunk_size = 300, chunk_overlap = 50)
chunks = splitter.split_documents(docs)
chunks

[Document(metadata={'source': '/Users/pawanpahune/RAG_From_Scratch/Query_Enhancement_Techniques/Langchain_crewai.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use'),
 Document(metadata={'source': '/Users/pawanpahune/RAG_From_Scratch/Query_Enhancement_Techniques/Langchain_crewai.txt'}, page_content='and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)'),
 Document(metadata={'source': '/Users/pawanpahune/RAG_From_Scratch/Query_Enhancement_Techniques/Langchain_crewai.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can 

In [9]:
embedding_model = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding_model)

retriever = vectorstore.as_retriever(search_type = "mmr", search_kwargs = {"k":5})

retriever


/var/folders/ly/dps_wwpj7972w23hbq80tr9r0000gn/T/ipykernel_25938/1527476905.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8185.02it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x119c0a110>, search_type='mmr', search_kwargs={'k': 5})

In [12]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

llm = init_chat_model('groq:openai/gpt-oss-120b', temperature=0.7)



In [14]:
query_expansion_prompt = PromptTemplate.from_template(
    """You are a helpful assistant. Your task is to expand the user's query to include more relevant keywords and phrases that can help retrieve more relevant documents.

    Original Query: "{query}"

    Expanded Query: """
)

query_expansion_chain = query_expansion_prompt|llm|StrOutputParser()

In [15]:
query_expansion_chain.invoke({"query": "Langchain_Memory"})

'**Expanded Query:**  \n\n`LangChain memory OR LangChain memory types OR LangChain conversation memory OR LangChain memory buffer OR LangChain memory management OR LangChain memory examples OR LangChain memory integration OR LangChain memory modules OR LangChain memory persistence OR LangChain memory serialization OR LangChain memory best practices OR LangChain memory tutorials OR LangChain memory agents OR LangChain memory vs other frameworks OR LangChain vectorstore memory OR LangChain chat memory OR LangChain memory retrieval OR LangChain memory patterns OR LangChain memory window OR LangChain memory store OR LangChain memory with OpenAI OR LangChain memory with LLM OR LangChain memory implementation in Python`'

In [16]:
question_prompt = PromptTemplate.from_template(
    """
    Answer the question based on the retrieved context.
    Context: {context}

    Question: {input}
    """
)

document_chain = create_stuff_documents_chain(llm = llm, prompt = question_prompt)

In [17]:
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\n    Answer the question based on the retrieved context.\n    Context: {context}\n\n    Question: {input}\n    ')
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x11a02e390>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x119f7f690>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))
| StrOutputParser(), kwargs={}, config={'run_name': 'stuff_documents_chain'}, confi

In [19]:
rag_pipeline = (
    RunnableMap(
        {
            "input": lambda x: x["input"],
            "context": lambda x: retriever.invoke(query_expansion_chain.invoke({"query": x["input"]}))
        }
    )
    | document_chain
)

In [20]:
rag_pipeline

{
  input: RunnableLambda(...),
  context: RunnableLambda(...)
}
| RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
    context: RunnableLambda(format_docs)
  }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
  | PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\n    Answer the question based on the retrieved context.\n    Context: {context}\n\n    Question: {input}\n    ')
  | ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x11a02e390>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x119f7f690>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))
  | StrO

In [24]:
query = {"input": "crew_ai agents"}
print(query_expansion_chain.invoke({"query": query}))
response = rag_pipeline.invoke(query)
print("Answer is following :",response)

**Expanded Query**

```
crew_ai agents
OR crew AI agents
OR crewAI agents
OR autonomous AI agents
OR multi‑agent systems
OR collaborative AI agents
OR AI crew management
OR LLM agents
OR LangChain agents
OR AutoGPT
OR AI agent framework
OR crew AI platform
OR crew AI SDK
OR crew AI documentation
OR crew AI examples
OR crew AI tutorials
OR crew AI use cases
OR crew AI deployment
OR crew AI integration
OR crew AI orchestration
OR AI team assistants
OR AI workflow agents
OR AI task automation agents
```
Answer is following : CrewAI agents are autonomous “team‑members” that work together in a structured workflow.  
Each agent is given a specific role—such as **researcher**, **planner**, **executor**, or any custom role you define—and operates semi‑independently while collaborating with the other agents in the crew.  

Developers describe a crew in a YAML or JSON‑style configuration, specifying for each agent:

* **Role** – what the agent is responsible for (e.g., gathering data, creating a